# Daphnet FoG Detection — Improved Pipeline
## Part 2: LOSO Evaluation Pipeline

Compares 6 classifiers using Leave-One-Subject-Out cross-validation:
- RandomForest, LogisticRegression, SVM, MLP, AdaBoost, XGBoost

In [1]:
from __future__ import annotations

import sys, os, time, json, warnings, logging, pickle
from pathlib import Path
from typing import Dict, List, Tuple, Any, Optional

import numpy as np
import pandas as pd
from scipy.signal import butter, sosfiltfilt
from scipy.optimize import minimize
from joblib import Parallel, delayed
from tqdm import tqdm

from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.impute import KNNImputer
from sklearn.feature_selection import SelectKBest, mutual_info_classif, VarianceThreshold
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, precision_score,
                             recall_score, f1_score, roc_auc_score, confusion_matrix, roc_curve)
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier

warnings.filterwarnings("ignore")

def find_project_root() -> Path:
    cwd = Path.cwd().resolve()
    candidates = [cwd, *cwd.parents]
    for candidate in candidates:
        if (candidate / "loaders").is_dir() and (candidate / "scripts").is_dir() and (candidate / "features").is_dir():
            return candidate
    raise RuntimeError("Could not locate project root from current working directory")

PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from scripts.utils.pipeline_utils import (
    get_classifiers, get_param_grids, prepare_fold, preprocess_features,
    train_and_evaluate_classifier, build_base_model, aggregate_results,
    print_results_table, print_fusion_results, youden_threshold, compute_metrics,
    HAS_XGB, HAS_SMOTE,
)

try:
    from xgboost import XGBClassifier
except ImportError:
    pass

try:
    from imblearn.over_sampling import SMOTE
except ImportError:
    pass

logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s", datefmt="%H:%M:%S")
log = logging.getLogger("daphnet")

# ── Constants ──
FS = 64
WINDOW_SEC = 4.0
WINDOW_SAMPLES = int(WINDOW_SEC * FS)
TRAIN_OVERLAP = 0.50
TEST_OVERLAP = 0.0
LABEL_THRESH = 0.50
BP_LOW, BP_HIGH, BP_ORDER = 0.5, 20.0, 4
NPERSEG = min(128, WINDOW_SAMPLES)
K_FEATURES = 60
SEED = 42
N_INNER_CV = 3
N_SEARCH_ITER = 20
ZERO_FOG_SUBJECTS = {4, 10}

OUTPUT_DIR = PROJECT_ROOT / "outputs" / "daphnet_improved_results"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

t0 = time.time()

# Load features from notebook 07
features_path = OUTPUT_DIR / "features.pkl"
with open(features_path, "rb") as f:
    features = pickle.load(f)
log.info("Loaded features for %d subjects from %s", len(features), features_path)

02:57:29 [INFO] Loaded features for 10 subjects from C:\Users\david\Desktop\Experimentos\outputs\daphnet_improved_results\features.pkl


In [2]:
def run_loso_evaluation(features: Dict):
    """Run full LOSO evaluation for all classifiers."""
    classifiers = get_classifiers(SEED)
    param_grids = get_param_grids()
    subjects = sorted(features.keys())

    all_results = {name: [] for name in classifiers}

    for test_sid in tqdm(subjects, desc="LOSO folds"):
        X_train, y_train, X_test, y_test = prepare_fold(features, test_sid)

        # Check if test fold has both classes
        has_both = len(np.unique(y_test)) == 2
        n_fog = int(np.sum(y_test == 1))

        log.info("Fold S%02d: test=%d samples (%d FoG), train=%d samples, both_classes=%s",
                 test_sid, len(y_test), n_fog, len(y_train), has_both)

        # Preprocess
        X_train_p, X_test_p, sel_cols, pipes = preprocess_features(X_train, X_test, y_train, k=K_FEATURES)

        # Train classifiers in parallel
        def _train_clf(clf_name):
            clf = get_classifiers(SEED)[clf_name]
            grid = param_grids[clf_name]
            m = train_and_evaluate_classifier(clf_name, clf, grid, X_train_p, y_train,
                                               X_test_p, y_test, seed=SEED,
                                               n_inner_cv=N_INNER_CV, n_search_iter=N_SEARCH_ITER,
                                               fold_info=f"S{test_sid:02d}")
            m["subject"] = test_sid
            m["has_both_classes"] = has_both
            return clf_name, m

        fold_results = Parallel(n_jobs=-1, verbose=0)(
            delayed(_train_clf)(name) for name in classifiers
        )

        for clf_name, m in fold_results:
            all_results[clf_name].append(m)

    return all_results

In [3]:
# Step 3: LOSO evaluation
log.info("Running LOSO evaluation with %d classifiers...", len(get_classifiers(SEED)))
all_results = run_loso_evaluation(features)

02:57:29 [INFO] Running LOSO evaluation with 5 classifiers...


LOSO folds:   0%|          | 0/10 [00:00<?, ?it/s]

02:57:29 [INFO] Fold S01: test=948 samples (45 FoG), train=7947 samples, both_classes=True


LOSO folds:  10%|█         | 1/10 [06:48<1:01:18, 408.77s/it]

03:04:18 [INFO] Fold S02: test=705 samples (92 FoG), train=8190 samples, both_classes=True


...

03:52:17 [INFO] Fold S09: test=869 samples (131 FoG), train=8026 samples, both_classes=True


LOSO folds:  90%|█████████ | 9/10 [1:00:50<06:30, 390.39s/it]

03:58:20 [INFO] Fold S10: test=1114 samples (0 FoG), train=7781 samples, both_classes=False


LOSO folds: 100%|██████████| 10/10 [1:06:20<00:00, 371.64s/it]

LOSO folds: 100%|██████████| 10/10 [1:06:20<00:00, 398.05s/it]

## Classifier Comparison Results

In [4]:
# Step 4: Print classifier results
clf_rows = print_results_table(all_results)


  CLASSIFIER COMPARISON
Classifier          F1(agg)  F1(mean)   Recall     Prec     Spec   BalAcc      AUC  Folds
------------------------------------------------------------------------------------------
AdaBoost             0.3617    0.4165   0.7425   0.3243   0.7565   0.7495   0.8361   8/10
RandomForest         0.3606    0.3221   0.5809   0.3776   0.8153   0.6981   0.8296   8/10
LogisticReg          0.3431    0.3048   0.6528   0.2270   0.7765   0.7147   0.7379   8/10
MLP                  0.3192    0.3180   0.5354   0.2783   0.8349   0.6851   0.7286   8/10
SVM                  0.2537    0.2836   0.5267   0.2654   0.8325   0.6796   0.7228   8/10
------------------------------------------------------------------------------------------

TOP 3 CLASSIFIERS (by aggregated F1):
  1. AdaBoost — F1=0.3617, AUC=0.8361
  2. RandomForest — F1=0.3606, AUC=0.8296
  3. LogisticReg — F1=0.3431, AUC=0.7379


In [5]:
# Save LOSO results for notebook 09
import pickle

loso_results_path = OUTPUT_DIR / "all_results.pkl"
with open(loso_results_path, "wb") as f:
    pickle.dump(all_results, f, protocol=pickle.HIGHEST_PROTOCOL)
log.info("LOSO results saved to %s", loso_results_path)

04:03:50 [INFO] LOSO results saved to C:\Users\david\Desktop\Experimentos\outputs\daphnet_improved_results\all_results.pkl
